In [2]:
from qubit import Qubit, invbell, xrotation, bell, invbell, zrot, xrot
from sympy import Symbol, Eq, expand, sin, cos
from sympy.physics.paulialgebra import Pauli, I
from sympy_pauli import evaluate_labelled_pauli_product

# Quantum Teleportation

Prepare qubits b and c as correlated in a bell state. Qubit b is the control and qubit c is the target, but the operation is close to  symmetrical; the inverse bell operation just makes their observables depend on the other qubit's observables.

Then Send qubit c a very long way away.

The state of another qubit, a, will eventually be transferred ("teleported") to qubit c.

In [3]:
(qubit_c_time1, qubit_b_time1) = invbell(Qubit.qubit_time_0("c"), Qubit.qubit_time_0("b"))

display(qubit_b_time1)
display(qubit_c_time1)

Qubit(b, b1, b2*c1, b3*c1)

Qubit(c, b1*c3, -b1*c2, c1)

Rotate qubit a around the x-axis by the secret θ to prepare the state that we will teleport.

In [4]:
θ = Symbol("theta")
qubit_a_time1 = xrotation(Qubit.qubit_time_0("a"), θ)
display(qubit_a_time1)

Qubit(a, a1, sin(theta)*a3 + cos(theta)*a2, -sin(theta)*a2 + cos(theta)*a3)

Then qubit b travels to qubit a and undergoes a Bell measurement (CNOT followed by Hadamard of control qubit) operation together with qubit a. qubit a is the control and qubit b is the target.

In [5]:
(qubit_a_time2, qubit_b_time2) = bell(qubit_a_time1, qubit_b_time1)
print("Qubits at time 2:")
display(qubit_a_time2)
display(qubit_b_time2)
display(qubit_c_time1)
print("The goal is to make qubit c perform exactly as qubit a did at time 1, just after the rotation:")
display(qubit_a_time1)

Qubits at time 2:


Qubit(a, -sin(theta)*a2 + cos(theta)*a3, -(sin(theta)*a3 + cos(theta)*a2)*b1, a1*b1)

Qubit(b, b1, (-sin(theta)*a2 + cos(theta)*a3)*b2*c1, (-sin(theta)*a2 + cos(theta)*a3)*b3*c1)

Qubit(c, b1*c3, -b1*c2, c1)

The goal is to make qubit c perform exactly as qubit a did at time 1, just after the rotation:


Qubit(a, a1, sin(theta)*a3 + cos(theta)*a2, -sin(theta)*a2 + cos(theta)*a3)

We're ready to measure qubit a in the computational basis, which means that we're interested in qubit a's Z observable. The initial X observable from both qubits a and b is maximally un-sharp, so it's an exactly 50/50 chance whether we'll see eigenvalue 1 or -1.

There are two outcomes which decohere from each other and we determine a fact about *this* decohered "world". This fact duplicates itself like a virus throughout all the "classical" machinery involved in the measurement. 

If we see <b>eigenvalue 1</b> for joint observable a<sub>1</sub>b<sub>1</sub> then this means that <b>a<sub>1</sub> = b<sub>1</sub></b>.

If we see <b>eigenvalue -1</b> for joint observable a<sub>1</sub>b<sub>1</sub> then this means that <b>a<sub>1</sub> = -b<sub>1</sub></b>. They have opposite phases.

<b>TODO: Need the maths on how the joint observable changes to one of the projectors in order to prove these statements! ^^^</b> https://algassert.com/quantum/2016/01/19/unknown-but-equal.html

In the Everettian view of quantum mechanics, both of these outcomes happen and there's a version of us which sees eigenvalue 1 and a version of us which sees eigenvalue -1 as a result of the measurement.

Then we can measure qubit b in the computational basis. The Z observable of b has value: (−sin(θ)a<sub>2</sub> + cos(θ)a<sub>3</sub>)b<sub>3</sub>c<sub>1</sub>.

Because we've setup the Schrödinger state of these qubits to be $\ket{000}$, we know that b<sub>3</sub> will always measure the eigenvalue 1. It can effectively be ignored.

If we see <b>eigenvalue 1</b> for joint observable (−sin(θ)a<sub>2</sub> + cos(θ)a<sub>3</sub>)c<sub>1</sub> then this means that <b>c<sub>1</sub> = (−sin(θ)a<sub>2</sub> + cos(θ)a<sub>3</sub>)</b>.

If we see <b>eigenvalue -1</b> for joint observable (−sin(θ)a<sub>2</sub> + cos(θ)a<sub>3</sub>)c<sub>1</sub> then this means that <b>c<sub>1</sub> = -(−sin(θ)a<sub>2</sub> + cos(θ)a<sub>3</sub>)</b>. They have opposite phases.

Here are the 4 situations which the environment around qubit c is in after its locality has been "infected" by the two "classical bits" sent over from the original location:

In [6]:
qubit_c_replaced_plus1_plus1 = qubit_c_time1.with_xz_observables(Pauli(1, "a"), (-sin(θ) * Pauli(2, "a") + cos(θ) * Pauli(3, "a")))
qubit_c_replaced_plus1_minus1 = qubit_c_time1.with_xz_observables(Pauli(1, "a"), -(-sin(θ) * Pauli(2, "a") + cos(θ) * Pauli(3, "a")))
qubit_c_replaced_minus1_plus1 = qubit_c_time1.with_xz_observables(-Pauli(1, "a"), (-sin(θ) * Pauli(2, "a") + cos(θ) * Pauli(3, "a")))
qubit_c_replaced_minus1_minus1 = qubit_c_time1.with_xz_observables(-Pauli(1, "a"), -(-sin(θ) * Pauli(2, "a") + cos(θ) * Pauli(3, "a")))

display(qubit_c_replaced_plus1_plus1)
display(qubit_c_replaced_plus1_minus1)
display(qubit_c_replaced_minus1_plus1)
display(qubit_c_replaced_minus1_minus1)

Qubit(c, a1, I*a1*(-sin(theta)*a2 + cos(theta)*a3), -sin(theta)*a2 + cos(theta)*a3)

Qubit(c, a1, -I*a1*(-sin(theta)*a2 + cos(theta)*a3), -(-sin(theta)*a2 + cos(theta)*a3))

Qubit(c, -a1, -I*a1*(-sin(theta)*a2 + cos(theta)*a3), -sin(theta)*a2 + cos(theta)*a3)

Qubit(c, -a1, I*a1*(-sin(theta)*a2 + cos(theta)*a3), -(-sin(theta)*a2 + cos(theta)*a3))

Applying the appropriate X and Z rotation gates to each situation will result in qubit c being in the same state as qubit a originally was after the rotation.

In [7]:
display(qubit_c_replaced_plus1_plus1)
display(xrot(qubit_c_replaced_plus1_minus1))
display(zrot(qubit_c_replaced_minus1_plus1))
display(zrot(xrot(qubit_c_replaced_minus1_minus1)))

Qubit(c, a1, I*a1*(-sin(theta)*a2 + cos(theta)*a3), -sin(theta)*a2 + cos(theta)*a3)

Qubit(c, a1, I*a1*(-sin(theta)*a2 + cos(theta)*a3), -sin(theta)*a2 + cos(theta)*a3)

Qubit(c, a1, I*a1*(-sin(theta)*a2 + cos(theta)*a3), -sin(theta)*a2 + cos(theta)*a3)

Qubit(c, a1, I*a1*(-sin(theta)*a2 + cos(theta)*a3), -sin(theta)*a2 + cos(theta)*a3)

TODO: I'd like these environmental "facts" to be recorded as part of a mesaurement process, allowing us to deal with each of the 4 scenarios without hardcoding things like `Pauli(1, "a"), (-sin(θ) * Pauli(2, "a") + cos(θ) * Pauli(3, "a"))` into this notebook.